# 04 - Google Play Store Preprocessing & Data Cleaning
Pipeline preprocessing terstandar, reproducible, dan privacy-reduced untuk dataset ulasan Google Play Store ekosistem aplikasi Zahir.


## 1. Objective & Scope
Tujuan dari notebook ini adalah:
1. Memverifikasi integritas dataset mentah (`playstore_reviews_raw.csv`) tanpa mengubah data aslinya (*raw data is immutable*).
2. Menerapkan strategi pengurangan privasi (*privacy-reduced analytical dataset*) dengan menghapus URL avatar pengguna dan menghasilkan `reviewer_id` berbasis hash SHA-256.
3. Melakukan pembersihan teks konservatif pada ulasan pengguna (`review_text_clean`) tanpa menghapus emoji, tanda baca berlebih, atau kata negasi.
4. Melakukan standardisasi datetime (`review_datetime`, `replied_datetime`) dan menghitung waktu respon pengembang (`response_time_days`).
5. Memetakan kategori rating (*rating category*: critical, neutral, positive) dengan batasan tegas bahwa rating numerik **bukan** pengganti sentimen teks.
6. Mengekspor dataset analitik final ke `data/final/playstore_reviews_final.csv` dan laporan missing values ke `data/final/playstore_missing_value_report.csv`.


## 2. Library Import & Environment Check
Mengimpor library komputasi data, regular expressions, modul hashing, serta modul custom preprocessing.


In [1]:
import os
import sys
import hashlib
import unicodedata
import pandas as pd
import numpy as np

# Tambahkan root direktori proyek ke sys.path untuk impor modular
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from preprocessing.playstore_preprocessing import (
    clean_review_text,
    map_rating_category,
    generate_reviewer_id,
    preprocess_playstore_dataset
)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Pandas version: 3.0.6
NumPy version: 2.4.6


## 3. Load Raw Dataset
Memuat dataset mentah dari `data/raw/playstore_reviews_raw.csv` dan memverifikasi integritas awal file.


In [2]:
raw_path = os.path.join('..', 'data', 'raw', 'playstore_reviews_raw.csv')
assert os.path.exists(raw_path), f"File mentah tidak ditemukan di {raw_path}!"

df_raw = pd.read_csv(raw_path)
print(f"Dataset mentah berhasil dimuat: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom.")
df_raw.head()


Dataset mentah berhasil dimuat: 417 baris, 13 kolom.


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name,package_name
0,db013820-004c-4ad3-82fd-4fb18de20967,Topik Nopal,https://play-lh.googleusercontent.com/a-/ALV-U...,"aplikasi tidak kompatibel terus dihp saya,kena...",5,0,NaN,2025-10-08 10:33:00,NaN,NaN,NaN,Zahir Attendance,com.zahir.attendance
1,8817939a-5285-4ebf-99cb-1a04b1ac2862,Sutrisman Daud,https://play-lh.googleusercontent.com/a/ACg8oc...,buruk,1,0,NaN,2025-10-06 00:59:53,Hai Sutrisman Daud. Kami mohon maaf atas kekec...,2026-01-02 17:36:32,NaN,Zahir Attendance,com.zahir.attendance
2,d21ab8dd-1042-4fca-8985-106d702cdb60,Po3y3 SE,https://play-lh.googleusercontent.com/a-/ALV-U...,Sering gagal Absensi setelah Update dengan ket...,3,0,2.0.40,2025-09-29 07:53:03,Hai Po3y3 SE. Kami mohon maaf atas kendala pre...,2026-01-02 17:38:21,2.0.40,Zahir Attendance,com.zahir.attendance
3,1d4508ff-30a4-4a18-b6f3-2aae38f5b9a7,Sutrisman Daud,https://play-lh.googleusercontent.com/a/ACg8oc...,"aplikasi macam apa ini, ga bisa absen. wajah t...",1,0,2.0.37,2025-08-28 19:16:10,Hai Sutrisman Daud. Kami mohon maaf atas kenda...,2026-01-02 17:35:15,2.0.37,Zahir Attendance,com.zahir.attendance
4,ddc0eb85-82f8-46da-a285-e735a90ae1b3,rino rino,https://play-lh.googleusercontent.com/a/ACg8oc...,Bagus dan mudah,5,0,2.0.36,2025-07-11 10:22:24,NaN,NaN,2.0.36,Zahir Attendance,com.zahir.attendance


## 4. Raw Data Validation & Schema Audit
Memeriksa skema 13 kolom mentah, keunikan `reviewId`, ketiadaan duplikasi baris penuh, serta distribusi aplikasi ekosistem Zahir.


In [3]:
expected_raw_cols = [
    'reviewId', 'userName', 'userImage', 'content', 'score',
    'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
    'repliedAt', 'appVersion', 'app_name', 'package_name'
]

print("=== VALIDASI SKEMA MENTAH ===")
print("Schema match 100%:", list(df_raw.columns) == expected_raw_cols)
print("Jumlah baris:", len(df_raw))
print("Jumlah reviewId unik:", df_raw['reviewId'].nunique())
print("Jumlah duplikasi baris penuh:", df_raw.duplicated().sum())
print("Jumlah konten ulasan kosong/null:", df_raw['content'].isna().sum())

print("\nDistribusi Aplikasi Ekosistem Zahir:")
print(df_raw['app_name'].value_counts())


=== VALIDASI SKEMA MENTAH ===
Schema match 100%: True
Jumlah baris: 417
Jumlah reviewId unik: 417
Jumlah duplikasi baris penuh: 0
Jumlah konten ulasan kosong/null: 0

Distribusi Aplikasi Ekosistem Zahir:
app_name
Zahir Simple Start     160
Zahir Online            98
POS X                   87
Zahir Certification     44
Zahir Attendance        28
Name: count, dtype: int64


## 5. Missing Value Audit (Raw Data)
Menginspeksi nilai hilang pada dataset mentah sebelum pemrosesan.


In [4]:
raw_missing = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing Count': df_raw.isna().sum().values,
    'Missing Percentage (%)': (df_raw.isna().sum().values / len(df_raw) * 100).round(2),
    'Data Type': [str(t) for t in df_raw.dtypes.values]
})
print("Laporan Missing Values Dataset Mentah:")
raw_missing


Laporan Missing Values Dataset Mentah:


,Column,Missing Count,Missing Percentage (%),Data Type
0,reviewId,0,0.00,str
1,userName,0,0.00,str
2,userImage,0,0.00,str
3,content,0,0.00,str
4,score,0,0.00,int64
5,thumbsUpCount,0,0.00,int64
6,reviewCreatedVersion,105,25.18,str
7,at,0,0.00,str
8,replyContent,51,12.23,str
9,repliedAt,51,12.23,str


## 6. Privacy & Identifier Handling
Untuk mematuhi prinsip etika data:
- Kolom avatar pengguna (`userImage`) tidak disimpan pada dataset analitik.
- Nama akun mentah (`userName`) tidak dijadikan fitur analitik.
- Dibuat `reviewer_id` berbasis hash SHA-256 (`hash(userName + reviewId)`) sepanjang 16 karakter untuk menjaga relasi peninjau anonim tanpa mengekspos identitas pribadi.
- Dataset ini dikategorikan sebagai *privacy-reduced analytical dataset*.


In [5]:
df_proc = df_raw.copy()

# Pembuatan reviewer_id anonim
df_proc['reviewer_id'] = [
    generate_reviewer_id(un, rid)
    for un, rid in zip(df_proc['userName'], df_proc['reviewId'])
]
df_proc['review_id'] = df_proc['reviewId'].astype(str)

print("Contoh penyamaran identitas:")
df_proc[['review_id', 'userName', 'reviewer_id']].head()


Contoh penyamaran identitas:


,review_id,userName,reviewer_id
0,db013820-004c-4ad3-82fd-4fb18de20967,Topik Nopal,7f6e41d40600f801
1,8817939a-5285-4ebf-99cb-1a04b1ac2862,Sutrisman Daud,af1d0a946ef8d1b6
2,d21ab8dd-1042-4fca-8985-106d702cdb60,Po3y3 SE,c642b3ea5bfc0d15
3,1d4508ff-30a4-4a18-b6f3-2aae38f5b9a7,Sutrisman Daud,6ce02bef22933aaa
4,ddc0eb85-82f8-46da-a285-e735a90ae1b3,rino rino,28ef1b7190addab0


## 7. Text Cleaning (Conservative Strategy)
Pembersihan teks ulasan dilakukan secara konservatif:
- Normalisasi Unicode NFKC.
- Menghapus URL dan karakter kontrol tak terlihat.
- Normalisasi baris baru dan spasi berlebih menjadi spasi tunggal.
- **TIDAK** menghapus emoji, tanda baca ekspresi, kata negasi (*tidak, gak, nggak*), dan belum menerapkan stopword removal / stemming agar tidak mengaburkan konteks NLP berikutnya.
- Menghasilkan fitur kualitas teks: `text_length_chars` dan `word_count`.


In [6]:
df_proc['review_text'] = df_proc['content'].astype(str)
df_proc['review_text_clean'] = df_proc['content'].apply(clean_review_text)

df_proc['text_length_chars'] = df_proc['review_text_clean'].str.len()
df_proc['word_count'] = df_proc['review_text_clean'].apply(lambda x: len(x.split()) if len(x.strip()) > 0 else 0)

print("=== STATISTIK PANJANG TEKS ULASAN ===")
print("Karakter (text_length_chars):\n", df_proc['text_length_chars'].describe())
print("\nKata (word_count):\n", df_proc['word_count'].describe())

print("\nPerbandingan Teks Asli vs Teks Bersih:")
sample_idx = df_proc[df_proc['content'].str.contains(r'\n')].index[:3]
for idx in sample_idx:
    print(f"--- Review ID: {df_proc.loc[idx, 'review_id'][:12]} ---")
    print("ASLI   :", repr(df_proc.loc[idx, 'review_text']))
    print("BERSIH :", repr(df_proc.loc[idx, 'review_text_clean']))


=== STATISTIK PANJANG TEKS ULASAN ===
Karakter (text_length_chars):
 count    417.000000
mean      73.491607
std       91.645091
min        2.000000
25%       23.000000
50%       44.000000
75%       81.000000
max      604.000000
Name: text_length_chars, dtype: float64

Kata (word_count):
 count    417.000000
mean      11.714628
std       14.520794
min        1.000000
25%        4.000000
50%        7.000000
75%       13.000000
max       88.000000
Name: word_count, dtype: float64

Perbandingan Teks Asli vs Teks Bersih:
--- Review ID: d77fa9b7-6ff ---
ASLI   : 'Akan lebih sempurna lagi, jika dibuatkan fitur seperti kasir online seperti \nMOKA.'
BERSIH : 'Akan lebih sempurna lagi, jika dibuatkan fitur seperti kasir online seperti MOKA.'
--- Review ID: 77507f80-626 ---
ASLI   : 'Aplikasi pembukuan real time yang sangat menarik untuk para pebisnis, mudah \ndigunakan dan dapat terhubung dengan zahir online desktop'
BERSIH : 'Aplikasi pembukuan real time yang sangat menarik untuk para pebisnis

## 8. Datetime Transformations & Response Time Calculation
Memproses timestamp `at` dan `repliedAt` menjadi representasi Datetime yang konsisten:
- `review_datetime` (timestamp ulasan), `review_date` (YYYY-MM-DD), `review_year`, `review_month`.
- `has_developer_reply` (Boolean).
- `replied_datetime` (timestamp balasan developer).
- `response_time_days` = `(replied_datetime - review_datetime) / 86400`.
- **Catatan Investigasi Temporal**: Ditemukan 15 ulasan dengan nilai `response_time_days < 0`. Ini terjadi karena pengguna memperbarui/mengedit ulasan mereka di Play Store setelah developer memberikan balasan pada tanggal sebelumnya. Nilai ini didokumentasikan secara transparan tanpa manipulasi data.


In [7]:
df_proc['review_datetime'] = pd.to_datetime(df_proc['at'])
df_proc['review_date'] = df_proc['review_datetime'].dt.strftime('%Y-%m-%d')
df_proc['review_year'] = df_proc['review_datetime'].dt.year
df_proc['review_month'] = df_proc['review_datetime'].dt.month

df_proc['has_developer_reply'] = df_proc['replyContent'].notna() & (df_proc['replyContent'].astype(str).str.strip() != '')
df_proc['reply_text'] = df_proc['replyContent']
df_proc['replied_datetime'] = pd.to_datetime(df_proc['repliedAt'])

time_diff_days = (df_proc['replied_datetime'] - df_proc['review_datetime']).dt.total_seconds() / 86400.0
df_proc['response_time_days'] = np.where(df_proc['has_developer_reply'], time_diff_days, np.nan)

print("=== DISTRIBUSI WAKTU RESPON DEVELOPER (HARI) ===")
print("Jumlah ulasan yang dibalas:", df_proc['has_developer_reply'].sum())
print("Deskripsi response_time_days:\n", df_proc['response_time_days'].describe())

neg_cases = df_proc[df_proc['response_time_days'] < 0]
print(f"\nJumlah kasus respon bernilai negatif (ulasan diedit pasca-balasan): {len(neg_cases)}")
neg_cases[['review_id', 'review_datetime', 'replied_datetime', 'response_time_days']].head()


=== DISTRIBUSI WAKTU RESPON DEVELOPER (HARI) ===
Jumlah ulasan yang dibalas: 366
Deskripsi response_time_days:
 count     366.000000
mean       19.904795
std       110.565075
min     -1016.938519
25%         0.015744
50%         0.190255
75%         2.239910
max       893.994363
Name: response_time_days, dtype: float64

Jumlah kasus respon bernilai negatif (ulasan diedit pasca-balasan): 15


,review_id,review_datetime,replied_datetime,response_time_days
18,950c3125-2b93-4b83-a6f8-3189b2119537,2020-06-03 08:05:27,2020-06-02 12:35:16,-0.812627
20,10279709-35e2-48df-9f14-6180cf81761f,2020-04-19 22:01:40,2020-04-19 21:08:37,-0.036840
44,255d1a6c-76aa-4576-a2bd-bbf3fad092f3,2020-09-03 15:14:37,2020-08-27 08:20:58,-7.287257
75,0e26fc8c-1186-4e3c-a7d7-71e1800a552e,2024-09-26 08:24:06,2021-12-14 09:52:38,-1016.938519
107,e0c075ae-283d-48d9-b2a9-ebdbbb3d4146,2020-04-04 15:10:39,2020-04-02 16:49:02,-1.931678


## 9. Rating Feature Engineering
Memetakan skor rating menjadi kategori bantuan:
- 1-2: `critical`
- 3: `neutral`
- 4-5: `positive`

**PENEGASAN METODOLOGI**:
Variabel `rating_category` adalah kategorisasi berbasis skor numerik bintang, **BUKAN** klasifikasi sentimen berbasis teks. Rating sentimen teks riil akan dieksplorasi secara terpisah pada fase analisis sentimen lanjutan.


In [8]:
df_proc['score'] = df_proc['score'].astype(int)
df_proc['rating_category'] = df_proc['score'].apply(map_rating_category)
df_proc['thumbs_up_count'] = df_proc['thumbsUpCount'].fillna(0).astype(int)

print("=== DISTRIBUSI RATING DAN KATEGORI ===")
print(pd.crosstab(df_proc['score'], df_proc['rating_category'], margins=True))


=== DISTRIBUSI RATING DAN KATEGORI ===
rating_category  critical  neutral  positive  All
score                                            
1                      74        0         0   74
2                      27        0         0   27
3                       0       37         0   37
4                       0        0        31   31
5                       0        0       248  248
All                   101       37       279  417


## 10. Application & Version Metadata
Mempertahankan identitas aplikasi dan metadata versi:
- Lima aplikasi ekosistem Zahir tetap dibedakan secara terpisah (`app_name` dan `package_name`).
- `reviewCreatedVersion` dan `appVersion` dipertahankan keduanya untuk ketertelusuran diagnostik versi.


In [9]:
df_proc['app_name'] = df_proc['app_name'].astype(str)
df_proc['package_name'] = df_proc['package_name'].astype(str)
df_proc['review_created_version'] = df_proc['reviewCreatedVersion']
df_proc['app_version'] = df_proc['appVersion']

print("Distribusi Ulasan per Aplikasi:")
print(df_proc['app_name'].value_counts())


Distribusi Ulasan per Aplikasi:
app_name
Zahir Simple Start     160
Zahir Online            98
POS X                   87
Zahir Certification     44
Zahir Attendance        28
Name: count, dtype: int64


## 11. Final Dataset Assembly & Schema
Menyusun 21 kolom analitik terstandar:
1. `review_id`
2. `reviewer_id`
3. `review_datetime`
4. `review_date`
5. `review_year`
6. `review_month`
7. `review_text`
8. `review_text_clean`
9. `score`
10. `rating_category`
11. `thumbs_up_count`
12. `has_developer_reply`
13. `reply_text`
14. `replied_datetime`
15. `response_time_days`
16. `app_name`
17. `package_name`
18. `review_created_version`
19. `app_version`
20. `text_length_chars`
21. `word_count`


In [10]:
final_cols = [
    'review_id', 'reviewer_id', 'review_datetime', 'review_date',
    'review_year', 'review_month', 'review_text', 'review_text_clean',
    'score', 'rating_category', 'thumbs_up_count', 'has_developer_reply',
    'reply_text', 'replied_datetime', 'response_time_days', 'app_name',
    'package_name', 'review_created_version', 'app_version',
    'text_length_chars', 'word_count'
]

df_final = df_proc[final_cols].copy()
print(f"Shape dataset final: {df_final.shape}")
df_final.head()


Shape dataset final: (417, 21)


,review_id,reviewer_id,review_datetime,review_date,review_year,review_month,review_text,review_text_clean,score,rating_category,...,has_developer_reply,reply_text,replied_datetime,response_time_days,app_name,package_name,review_created_version,app_version,text_length_chars,word_count
0,db013820-004c-4ad3-82fd-4fb18de20967,7f6e41d40600f801,2025-10-08 10:33:00,2025-10-08,2025,10,"aplikasi tidak kompatibel terus dihp saya,kena...","aplikasi tidak kompatibel terus dihp saya,kena...",5,positive,...,False,NaN,NaT,NaN,Zahir Attendance,com.zahir.attendance,NaN,NaN,52,7
1,8817939a-5285-4ebf-99cb-1a04b1ac2862,af1d0a946ef8d1b6,2025-10-06 00:59:53,2025-10-06,2025,10,buruk,buruk,1,critical,...,True,Hai Sutrisman Daud. Kami mohon maaf atas kekec...,2026-01-02 17:36:32,88.692118,Zahir Attendance,com.zahir.attendance,NaN,NaN,5,1
2,d21ab8dd-1042-4fca-8985-106d702cdb60,c642b3ea5bfc0d15,2025-09-29 07:53:03,2025-09-29,2025,9,Sering gagal Absensi setelah Update dengan ket...,Sering gagal Absensi setelah Update dengan ket...,3,neutral,...,True,Hai Po3y3 SE. Kami mohon maaf atas kendala pre...,2026-01-02 17:38:21,95.406458,Zahir Attendance,com.zahir.attendance,2.0.40,2.0.40,263,38
3,1d4508ff-30a4-4a18-b6f3-2aae38f5b9a7,6ce02bef22933aaa,2025-08-28 19:16:10,2025-08-28,2025,8,"aplikasi macam apa ini, ga bisa absen. wajah t...","aplikasi macam apa ini, ga bisa absen. wajah t...",1,critical,...,True,Hai Sutrisman Daud. Kami mohon maaf atas kenda...,2026-01-02 17:35:15,126.929919,Zahir Attendance,com.zahir.attendance,2.0.37,2.0.37,61,10
4,ddc0eb85-82f8-46da-a285-e735a90ae1b3,28ef1b7190addab0,2025-07-11 10:22:24,2025-07-11,2025,7,Bagus dan mudah,Bagus dan mudah,5,positive,...,False,NaN,NaT,NaN,Zahir Attendance,com.zahir.attendance,2.0.36,2.0.36,15,3


## 12. Export Processed Dataset & Reports
Mengekspor dataset hasil olahan ke `data/final/playstore_reviews_final.csv` dan laporan missing values ke `data/final/playstore_missing_value_report.csv`.


In [11]:
final_csv_path = os.path.join('..', 'data', 'final', 'playstore_reviews_final.csv')
report_csv_path = os.path.join('..', 'data', 'final', 'playstore_missing_value_report.csv')

os.makedirs(os.path.dirname(final_csv_path), exist_ok=True)

df_final.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
print(f"Dataset analitik berhasil disimpan ke: {final_csv_path}")

missing_report = pd.DataFrame({
    'column': df_final.columns,
    'missing_count': df_final.isna().sum().values,
    'missing_percentage': (df_final.isna().sum().values / len(df_final) * 100).round(2),
    'non_null_count': df_final.notna().sum().values,
    'dtype': [str(t) for t in df_final.dtypes.values]
})
missing_report.to_csv(report_csv_path, index=False)
print(f"Laporan missing values berhasil disimpan ke: {report_csv_path}")


Dataset analitik berhasil disimpan ke: ..\data\final\playstore_reviews_final.csv
Laporan missing values berhasil disimpan ke: ..\data\final\playstore_missing_value_report.csv


## 13. Comprehensive Verification & Quality Assurance
Menjalankan pengujian mutu menyeluruh pada file CSV final yang baru diekspor:
1. Pembacaan ulang file CSV.
2. Keunikan baris dan `review_id`.
3. Validasi nilai rating (1-5) dan kategori rating.
4. Validasi ketiadaan teks kosong pada `review_text_clean`.
5. Konsistensi status balasan developer.


In [12]:
df_check = pd.read_csv(final_csv_path, encoding='utf-8-sig')

assert len(df_check) == 417, f"Ekspektasi 417 baris, ditemukan {len(df_check)}"
assert len(df_check.columns) == 21, f"Ekspektasi 21 kolom, ditemukan {len(df_check.columns)}"
assert df_check['review_id'].duplicated().sum() == 0, "Ditemukan duplikasi review_id!"
assert df_check['score'].between(1, 5).all(), "Terdapat nilai score di luar 1-5!"
assert (df_check['review_text_clean'].str.strip() == '').sum() == 0, "Terdapat review_text_clean kosong!"
assert (df_check['has_developer_reply'] == df_check['reply_text'].notna()).all(), "Inkonsistensi flag has_developer_reply!"

print("SELURUH 6 ASSERTION PENGUJIAN MUTU LOLOS 100%!")


SELURUH 6 ASSERTION PENGUJIAN MUTU LOLOS 100%!


## 14. Summary & Methodological Limitations
### Ringkasan Hasil Preprocessing Fase 2:
- **Total ulasan**: 417 ulasan (100% dipertahankan, 0 baris dihapus).
- **Skema data**: Diperkaya dari 13 kolom mentah menjadi 21 kolom analitik terstruktur.
- **Privasi**: Avatar dihapus, nama di-hash menjadi `reviewer_id` sepanjang 16 karakter.
- **Pembersihan teks**: Normalisasi whitespace dan pembersihan URL tanpa merusak semantik ulasan.

### Keterbatasan Metodologi:
1. *Rating Category != Sentiment*: Kategori rating `critical`, `neutral`, `positive` mencerminkan skor bintang, bukan polaritas sentimen kalimat ulasan.
2. *Respon Negatif (15 Kasus)*: Selisih tanggal balasan developer yang lebih awal dari tanggal ulasan terjadi karena pengguna memperbarui ulasan setelah developer merespons.
3. *Missing Versions (105 Kasus)*: Informasi versi aplikasi tidak tersedia pada 25,18% ulasan karena platform Play Store mengizinkan pemberian ulasan tanpa mencantumkan versi rilis.
